## Метод потенциалов

In [ ]:
import numpy as np
from IPython.display import display, HTML, Markdown

def display_transportation_table(costs, basis_matrix, supply, demand, title=""):
    m, n = costs.shape
    
    html = f"<h3 style='color: #e0e0e0; font-family: sans-serif; margin-top: 20px;'>{title}</h3>"
    html += "<table style='border-collapse: collapse; text-align: center; font-family: sans-serif; min-width: 500px; background-color: #1e1e1e; color: #e0e0e0; border: 1px solid #444;'>"
    
    html += "<tr style='background-color: #2d2d2d; border: 1px solid #444;'>"
    html += "<th style='border: 1px solid #444; padding: 10px; color: #ffffff;'>Пункты</th>"
    for j in range(n):
        html += f"<th style='border: 1px solid #444; padding: 10px; color: #ffffff;'>B<sub>{j+1}</sub></th>"
    html += "<th style='border: 1px solid #444; padding: 10px; color: #ffffff;'>Запасы</th>"
    html += "</tr>"
    
    for i in range(m):
        html += "<tr>"
        html += f"<td style='border: 1px solid #444; font-weight: bold; background-color: #2d2d2d; color: #ffffff; padding: 10px;'>A<sub>{i+1}</sub></td>"
        
        for j in range(n):
            val = basis_matrix[i, j]
            if val is None:
                display_val = ""
            elif val == 0:
                display_val = "•"
            else:
                display_val = f"{int(val)}"
                
            cell_html = (
                f"<td style='border: 1px solid #444; width: 90px; height: 55px; position: relative; padding: 2px; background-color: #1e1e1e;'>"
                f"<div style='position: absolute; top: 3px; left: 6px; font-size: 11px; color: #888888;'>{int(costs[i,j])}</div>"
                f"<div style='position: absolute; bottom: 3px; right: 10px; font-weight: bold; font-size: 16px; color: #ffffff;'>{display_val}</div>"
                f"</td>"
            )
            html += cell_html
            
        html += f"<td style='border: 1px solid #444; font-weight: bold; background-color: #252525;'>{int(supply[i])}</td>"
        html += "</tr>"
        
    html += "<tr style='background-color: #2d2d2d; font-weight: bold;'>"
    html += "<td style='border: 1px solid #444; padding: 10px; color: #ffffff;'>Потребности</td>"
    for j in range(n):
        html += f"<td style='border: 1px solid #444;'>{int(demand[j])}</td>"
    html += f"<td style='border: 1px solid #444; background-color: #3d3d3d; color: #ffffff;'>{int(sum(supply))}</td>"
    html += "</tr>"
    html += "</table>"
    
    display(HTML(html))

def solve_transportation_universal(costs, supply, demand):
    costs = np.array(costs, dtype=float)
    supply = np.array(supply, dtype=float)
    demand = np.array(demand, dtype=float)
    
    orig_m, orig_n = costs.shape
    sum_supply = sum(supply)
    sum_demand = sum(demand)
    
    added_row = False
    added_col = False
    
    if sum_supply > sum_demand:
        diff = sum_supply - sum_demand
        costs = np.hstack((costs, np.zeros((orig_m, 1))))
        demand = np.append(demand, diff)
        added_col = True
    elif sum_supply < sum_demand:
        diff = sum_demand - sum_supply
        costs = np.vstack((costs, np.zeros((1, orig_n))))
        supply = np.append(supply, diff)
        added_row = True
        
    m, n = costs.shape
    
    basis_matrix = np.full((m, n), None, dtype=object)
    s = supply.copy()
    d = demand.copy()
    i, j = 0, 0
    while i < m and j < n:
        val = min(s[i], d[j])
        basis_matrix[i, j] = val
        s[i] -= val
        d[j] -= val
        if s[i] == 0 and i < m - 1 and (d[j] != 0 or j == n - 1):
            i += 1
        elif d[j] == 0 and j < n - 1:
            j += 1
        else:
            i += 1
            j += 1

    iteration = 0
    while True:
        iteration += 1
        title_str = f"Итерация {iteration}"
        display_transportation_table(costs, basis_matrix, supply, demand, title=title_str)
        
        alpha, beta = [None] * m, [None] * n
        alpha[0] = 0.0
        changed = True
        while changed:
            changed = False
            for r in range(m):
                for c in range(n):
                    if basis_matrix[r, c] is not None:
                        if alpha[r] is not None and beta[c] is None:
                            beta[c] = costs[r, c] - alpha[r]
                            changed = True
                        elif beta[c] is not None and alpha[r] is None:
                            alpha[r] = costs[r, c] - beta[c]
                            changed = True
                            
        optimal = True
        min_delta = 0.0
        target_cell = None
        for r in range(m):
            for c in range(n):
                if basis_matrix[r, c] is None:
                    delta = costs[r, c] - (alpha[r] + beta[c])
                    if delta < 0:
                        optimal = False
                        if delta < min_delta:
                            min_delta = delta
                            target_cell = (r, c)
                            
        if optimal:
            break
            
        cycle = find_cycle(basis_matrix, target_cell)
        minus_cells = cycle[1::2]
        theta = min(basis_matrix[r, c] for r, c in minus_cells)
        
        for idx, (r, c) in enumerate(cycle):
            if basis_matrix[r, c] is None: 
                basis_matrix[r, c] = 0.0
            if idx % 2 == 0:
                basis_matrix[r, c] += theta
            else:
                basis_matrix[r, c] -= theta
                
        removed = False
        for r, c in minus_cells:
            if basis_matrix[r, c] == 0 and not removed:
                basis_matrix[r, c] = None
                removed = True

    total_cost = 0
    equation_parts = []
    
    for r in range(m):
        for c in range(n):
            if basis_matrix[r, c] is not None:
                val = basis_matrix[r, c]
                if (added_row and r == m - 1) or (added_col and c == n - 1):
                    continue
                if val > 0:
                    total_cost += val * costs[r, c]
                    equation_parts.append(f"{int(costs[r,c])}·{int(val)}")
                
    equation_str = " + ".join(equation_parts)
    display(HTML("<h4 style='color: #ffffff;'>Оптимальный план найден</h4>"))
    display(Markdown(f"$f = {equation_str} = {int(total_cost)}$"))

def find_cycle(basis, start):
    m, n = basis.shape
    points = [(r, c) for r in range(m) for c in range(n) if basis[r, c] is not None]
    if start not in points: points.append(start)
    path = [start]

    def dfs(curr, move_row):
        nodes = [p for p in points if (p[0] == curr[0] if move_row else p[1] == curr[1]) and p != curr]
        for nxt in nodes:
            if nxt == start and len(path) >= 4 and len(path) % 2 == 0: return True
            if nxt not in path:
                path.append(nxt)
                if dfs(nxt, not move_row): return True
                path.pop()
        return False

    dfs(start, True)
    return path

In [26]:
display(Markdown("#### Сбалансированный"))
c1 = [[9, 5, 7, 10, 18], 
      [36, 29, 6, 38, 40],
      [41, 20, 11, 25, 19],
      [30, 28, 13, 39, 50]]
s1 = [78, 94, 29, 86]
d1 = [49, 60, 78, 50, 50]

solve_transportation_universal(c1, s1, d1)

#### Сбалансированный

Пункты,B1,B2,B3,B4,B5,Запасы
A{i+1},949,529,7,10,18,78
A{i+1},36,29,678,38,4016,94
A{i+1},41,20,11,25,1929,29
A{i+1},30,2831,13,3950,505,86
Потребности,49,60,78,50,50,287


Пункты,B1,B2,B3,B4,B5,Запасы
A{i+1},949,524,7,10,185,78
A{i+1},36,29,678,38,4016,94
A{i+1},41,20,11,25,1929,29
A{i+1},30,2836,13,3950,50,86
Потребности,49,60,78,50,50,287


Пункты,B1,B2,B3,B4,B5,Запасы
A{i+1},949,5,7,1024,185,78
A{i+1},36,29,678,38,4016,94
A{i+1},41,20,11,25,1929,29
A{i+1},30,2860,13,3926,50,86
Потребности,49,60,78,50,50,287


Пункты,B1,B2,B3,B4,B5,Запасы
A{i+1},923,5,7,1050,185,78
A{i+1},36,29,678,38,4016,94
A{i+1},41,20,11,25,1929,29
A{i+1},3026,2860,13,39,50,86
Потребности,49,60,78,50,50,287


Пункты,B1,B2,B3,B4,B5,Запасы
A{i+1},9,523,7,1050,185,78
A{i+1},36,29,678,38,4016,94
A{i+1},41,20,11,25,1929,29
A{i+1},3049,2837,13,39,50,86
Потребности,49,60,78,50,50,287


$f = 5·23 + 10·50 + 18·5 + 6·78 + 40·16 + 19·29 + 30·49 + 28·37 = 4870$

In [ ]:
display(Markdown("#### С нарушением баланса"))
c2 = [[1, 2, 3], 
      [2, 3, 3]]
s2 = [20, 40]
d2 = [30, 30, 20]

solve_transportation_universal(c2, s2, d2)

## Метод Фогеля

In [43]:
import numpy as np
import pandas as pd
from IPython.display import display, HTML, Markdown

def display_transportation_table(costs, basis_matrix, supply, demand, title=""):
    m, n = costs.shape
    
    html = f"<h3 style='color: #ffffff; font-family: sans-serif; margin-top: 20px;'>{title}</h3>"
    html += "<table style='border-collapse: collapse; text-align: center; font-family: sans-serif; min-width: 500px; background-color: #1e1e1e; color: #e0e0e0; border: 1px solid #444;'>"
    
    html += "<tr style='background-color: #2d2d2d; border: 1px solid #444;'>"
    html += "<th style='border: 1px solid #444; padding: 10px; color: #ffffff;'>Пункты</th>"
    for j in range(n):
        html += f"<th style='border: 1px solid #444; padding: 10px; color: #ffffff;'>B<sub>{j+1}</sub></th>"
    html += "<th style='border: 1px solid #444; padding: 10px; color: #ffffff;'>Запасы</th>"
    html += "</tr>"
    
    for i in range(m):
        html += "<tr>"
        html += f"<td style='border: 1px solid #444; font-weight: bold; background-color: #2d2d2d; color: #ffffff; padding: 10px;'>A<sub>{i+1}</sub></td>"
        
        for j in range(n):
            val = basis_matrix[i, j]
            if val is None:
                display_val = ""
            elif val == 0:
                display_val = "•"
            else:
                display_val = f"{int(val)}"
                
            cell_html = (
                f"<td style='border: 1px solid #444; width: 90px; height: 55px; position: relative; padding: 2px; background-color: #1e1e1e;'>"
                f"<div style='position: absolute; top: 3px; left: 6px; font-size: 11px; color: #888888;'>{int(costs[i,j])}</div>"
                f"<div style='position: absolute; bottom: 3px; right: 10px; font-weight: bold; font-size: 16px; color: #ffffff;'>{display_val}</div>"
                f"</td>"
            )
            html += cell_html
            
        html += f"<td style='border: 1px solid #444; font-weight: bold; background-color: #252525;'>{int(supply[i])}</td>"
        html += "</tr>"
        
    html += "<tr style='background-color: #2d2d2d; font-weight: bold;'>"
    html += "<td style='border: 1px solid #444; padding: 10px; color: #ffffff;'>Потребности</td>"
    for j in range(n):
        html += f"<td style='border: 1px solid #444;'>{int(demand[j])}</td>"
    html += f"<td style='border: 1px solid #444; background-color: #3d3d3d; color: #ffffff;'>{int(sum(supply))}</td>"
    html += "</tr>"
    html += "</table>"
    
    display(HTML(html))

def vogel_initial_plan(costs, supply, demand):
    """
    Метод Фогеля для построения начального опорного плана.
    """
    m, n = costs.shape
    s = supply.copy().astype(float)
    d = demand.copy().astype(float)
    basis = np.full((m, n), None, dtype=object)
    
    active_rows = set(range(m))
    active_cols = set(range(n))
    
    while active_rows and active_cols:
        penalties = {}
        
        for r in active_rows:
            vals = [costs[r, c] for c in active_cols]
            penalties[('row', r)] = sorted(vals)[1] - sorted(vals)[0] if len(vals) > 1 else vals[0]
            
        for c in active_cols:
            vals = [costs[r, c] for r in active_rows]
            penalties[('col', c)] = sorted(vals)[1] - sorted(vals)[0] if len(vals) > 1 else vals[0]
            
        key = max(penalties, key=penalties.get)
        type_, idx = key
        
        if type_ == 'row':
            r = idx
            c = min(active_cols, key=lambda col: costs[r, col])
        else:
            c = idx
            r = min(active_rows, key=lambda row: costs[row, c])
            
        val = min(s[r], d[c])
        basis[r, c] = val
        s[r] -= val
        d[c] -= val
        
        if s[r] == 0 and len(active_rows) > 1:
            active_rows.remove(r)
        elif d[c] == 0 and len(active_cols) > 1:
            active_cols.remove(c)
        else:
            if r in active_rows: active_rows.remove(r)
            if c in active_cols: active_cols.remove(c)
            
    return basis

def find_cycle(basis, start):
    m, n = basis.shape
    points = [(r, c) for r in range(m) for c in range(n) if basis[r, c] is not None]
    if start not in points: points.append(start)
    path = [start]

    def dfs(curr, move_row):
        nodes = [p for p in points if (p[0] == curr[0] if move_row else p[1] == curr[1]) and p != curr]
        for nxt in nodes:
            if nxt == start and len(path) >= 4 and len(path) % 2 == 0: return True
            if nxt not in path:
                path.append(nxt)
                if dfs(nxt, not move_row): return True
                path.pop()
        return False

    dfs(start, True)
    return path

def solve_transportation_vogel_plus_potentials(costs, supply, demand):
    costs = np.array(costs, dtype=float)
    supply = np.array(supply, dtype=float)
    demand = np.array(demand, dtype=float)
    m, n = costs.shape
    
    basis_matrix = vogel_initial_plan(costs, supply, demand)
    
    iteration = 0
    while True:
        iteration += 1
        
        current_cost = 0
        for r in range(m):
            for c in range(n):
                if basis_matrix[r, c] is not None and basis_matrix[r, c] > 0:
                    current_cost += basis_matrix[r, c] * costs[r, c]
                    
        title_str = f"Итерация {iteration} (Текущая стоимость Z = {int(current_cost)})"
        if iteration == 1:
            title_str = f"Начальный план по методу Фогеля (Z = {int(current_cost)})"
            
        display_transportation_table(costs, basis_matrix, supply, demand, title=title_str)
        
        alpha, beta = [None] * m, [None] * n
        alpha[0] = 0.0
        
        while None in alpha or None in beta:
            changed = False
            for r in range(m):
                for c in range(n):
                    if basis_matrix[r, c] is not None:
                        if alpha[r] is not None and beta[c] is None:
                            beta[c] = costs[r, c] - alpha[r]
                            changed = True
                        elif beta[c] is not None and alpha[r] is None:
                            alpha[r] = costs[r, c] - beta[c]
                            changed = True
            
            if not changed and (None in alpha or None in beta):
                added = False
                for r in range(m):
                    for c in range(n):
                        if basis_matrix[r, c] is None:
                            if (alpha[r] is not None and beta[c] is None) or (beta[c] is not None and alpha[r] is None):
                                basis_matrix[r, c] = 0.0  
                                added = True
                                break
                    if added: break
                if not added: break

        optimal = True
        min_delta = 0.0
        target_cell = None
        for r in range(m):
            for c in range(n):
                if basis_matrix[r, c] is None:
                    delta = costs[r, c] - (alpha[r] + beta[c])
                    if delta < 0:  
                        optimal = False
                        if delta < min_delta:
                            min_delta = delta
                            target_cell = (r, c)
                            
        if optimal:
            break
            
        cycle = find_cycle(basis_matrix, target_cell)
        minus_cells = cycle[1::2]
        theta = min(basis_matrix[r, c] for r, c in minus_cells)
        
        for idx, (r, c) in enumerate(cycle):
            if basis_matrix[r, c] is None: 
                basis_matrix[r, c] = 0.0
            if idx % 2 == 0:
                basis_matrix[r, c] += theta
            else:
                basis_matrix[r, c] -= theta
                
        removed = False
        for r, c in minus_cells:
            if basis_matrix[r, c] == 0 and not removed:
                basis_matrix[r, c] = None
                removed = True

    total_cost = 0
    equation_parts = []
    for r in range(m):
        for c in range(n):
            if basis_matrix[r, c] is not None and basis_matrix[r, c] > 0:
                val = basis_matrix[r, c]
                total_cost += val * costs[r, c]
                equation_parts.append(f"{int(costs[r,c])}·{int(val)}")
                
    equation_str = " + ".join(equation_parts)
    display(HTML("<h4 style='color: #ffffff; margin-top: 20px;'>Оптимальный минимальный план найден</h4>"))
    display(Markdown(f"**Минимальная стоимость плана:** $f = {equation_str} = {int(total_cost)}$"))

costs = np.array([
    [9,  5,  7,  10, 18],
    [36, 29, 6,  38, 40],
    [41, 20, 11, 25, 19],
    [30, 28, 13, 39, 50]
])
supply = np.array([78, 94, 29, 86])
demand = np.array([49, 60, 78, 50, 50])

solve_transportation_vogel_plus_potentials(costs, supply, demand)

Пункты,B1,B2,B3,B4,B5,Запасы
A1,949,529,7,10,18,78
A2,36,29,678,38,4016,94
A3,41,20,11,25,1929,29
A4,30,2831,13,3950,505,86
Потребности,49,60,78,50,50,287


Пункты,B1,B2,B3,B4,B5,Запасы
A1,949,524,7,10,185,78
A2,36,29,678,38,4016,94
A3,41,20,11,25,1929,29
A4,30,2836,13,3950,50,86
Потребности,49,60,78,50,50,287


Пункты,B1,B2,B3,B4,B5,Запасы
A1,949,5,7,1024,185,78
A2,36,29,678,38,4016,94
A3,41,20,11,25,1929,29
A4,30,2860,13,3926,50,86
Потребности,49,60,78,50,50,287


Пункты,B1,B2,B3,B4,B5,Запасы
A1,923,5,7,1050,185,78
A2,36,29,678,38,4016,94
A3,41,20,11,25,1929,29
A4,3026,2860,13,39,50,86
Потребности,49,60,78,50,50,287


Пункты,B1,B2,B3,B4,B5,Запасы
A1,9,523,7,1050,185,78
A2,36,29,678,38,4016,94
A3,41,20,11,25,1929,29
A4,3049,2837,13,39,50,86
Потребности,49,60,78,50,50,287


**Минимальная стоимость плана:** $f = 5·23 + 10·50 + 18·5 + 6·78 + 40·16 + 19·29 + 30·49 + 28·37 = 4870$